# Módulo 13 — Projeto: Fundamentos da Descoberta de Dados

Análise da base de produtos de supermercado do Chile.

O notebook responde às 5 questões do projeto:
1. Média e mediana do `Preco_Normal` por categoria;
2. Desvio padrão do `Preco_Normal` por categoria;
3. Boxplot da categoria com maior desvio padrão;
4. Gráfico de barras da média de descontos por categoria;
5. Treemap interativo por categoria, marca e média de desconto.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

arquivos = sorted(Path(".").glob("dados_parte_*.csv"))
df = pd.concat([pd.read_csv(arq, delimiter=";") for arq in arquivos], ignore_index=True)

df.head(10)


## 1 — Média e mediana do `Preco_Normal` por categoria


In [ ]:
media_por_categoria = (
    df.groupby("Categoria")["Preco_Normal"]
      .mean()
      .sort_values(ascending=False)
)

media_por_categoria


In [ ]:
mediana_por_categoria = (
    df.groupby("Categoria")["Preco_Normal"]
      .median()
      .sort_values(ascending=False)
)

mediana_por_categoria


In [ ]:
comparacao = (
    df.groupby("Categoria")["Preco_Normal"]
      .agg(Media="mean", Mediana="median")
)

comparacao["Diferenca"] = comparacao["Media"] - comparacao["Mediana"]
comparacao.sort_values("Diferenca", ascending=False)


### Interpretação

Na maior parte das categorias, a **média está acima da mediana**, indicando que existem produtos com preços altos puxando a média para cima.

A principal exceção é **`comidas-preparadas`**, em que a média fica abaixo da mediana.

A diferença mais forte entre média e mediana ocorre em **`lacteos`**, o que já sugere uma distribuição bastante assimétrica.


## 2 — Desvio padrão por categoria


In [ ]:
desvio_padrao = (
    df.groupby("Categoria")["Preco_Normal"]
      .std()
      .sort_values(ascending=False)
)

desvio_padrao


In [ ]:
resumo_estatistico = (
    df.groupby("Categoria")["Preco_Normal"]
      .agg(Media="mean", Mediana="median", Desvio_Padrao="std")
      .sort_values("Desvio_Padrao", ascending=False)
)

resumo_estatistico


### Interpretação

A categoria com maior desvio padrão é **`lacteos`**, com desvio padrão de aproximadamente **3925.82**.

Nessa categoria, a média (**2385.22**) é bem maior que a mediana (**989.00**).

Isso mostra que a distribuição possui grande dispersão e que preços elevados estão puxando a média para cima.


## 3 — Boxplot da categoria com maior desvio padrão


In [ ]:
categoria_maior_desvio = "lacteos"

dados_categoria = df.loc[
    df["Categoria"] == categoria_maior_desvio,
    "Preco_Normal"
]

plt.figure(figsize=(10, 6))
plt.boxplot(dados_categoria, vert=False)
plt.title(f"Distribuição de Preco_Normal — {categoria_maior_desvio}")
plt.xlabel("Preço normal")
plt.show()


In [ ]:
q1 = dados_categoria.quantile(0.25)
q3 = dados_categoria.quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = dados_categoria[
    (dados_categoria < limite_inferior) |
    (dados_categoria > limite_superior)
]

print("Q1:", q1)
print("Q3:", q3)
print("Limite superior:", limite_superior)
print("Quantidade de outliers:", len(outliers))


### Interpretação do boxplot

A distribuição de **`lacteos`** é bastante assimétrica à direita.

Pelo critério de 1,5 × IQR, foram identificados **43 outliers** em 447 produtos da categoria.

O limite superior calculado foi **6119.00**. Os valores acima desse ponto aparecem como produtos de preço muito superior ao restante da categoria e ajudam a explicar por que a média é tão maior que a mediana.


## 4 — Média de descontos por categoria


In [ ]:
media_desconto = (
    df.groupby("Categoria", as_index=False)["Desconto"]
      .mean()
      .sort_values("Desconto", ascending=False)
)

media_desconto


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(media_desconto["Categoria"], media_desconto["Desconto"])
plt.title("Média de descontos por categoria")
plt.xlabel("Categoria")
plt.ylabel("Média do desconto")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Interpretação

A categoria com maior média de desconto é **`congelados`**, com média de aproximadamente **154.03**.

Na sequência aparece **`belleza-y-cuidado-personal`**, com média de aproximadamente **123.08**.


## 5 — Mapa interativo por categoria, marca e média de desconto


In [ ]:
mapa_descontos = (
    df.groupby(["Categoria", "Marca"], as_index=False)["Desconto"]
      .mean()
      .rename(columns={"Desconto": "Media_Desconto"})
)

fig = px.treemap(
    mapa_descontos,
    path=["Categoria", "Marca"],
    values="Media_Desconto",
    color="Media_Desconto",
    title="Média de desconto por categoria e marca"
)

fig.show()

# Salva uma visualização HTML interativa que pode ser publicada no GitHub Pages
fig.write_html("mapa_interativo_descontos.html")


## Conclusão

Os resultados mostram que:

- `lacteos` possui a maior dispersão de preços e grande diferença entre média e mediana;
- essa diferença é explicada por uma distribuição assimétrica e pela presença de vários produtos com preços muito altos;
- a média de desconto é mais elevada em `congelados`;
- o treemap permite explorar de maneira interativa quais categorias e marcas concentram maiores descontos.
